In [3]:
!pip install -q transformers torch pandas sentencepiece

In [4]:
import time
import torch
import pandas as pd

from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [5]:
bert = pipeline(
    "fill-mask",
    model="bert-base-uncased"
)

gpt2 = pipeline(
    "text-generation",
    model="gpt2"
)

t5_tok = AutoTokenizer.from_pretrained(
    "google/flan-t5-base"
)

t5 = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base"
).eval()

print("All three models loaded successfully!")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

... (19 outputs left out)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

All three models loaded successfully!


In [6]:
def run_bert(prompt):
    return bert(prompt)[0]["token_str"]


def run_gpt2(prompt):
    out = gpt2(
        prompt,
        max_new_tokens=12,
        do_sample=False,
        pad_token_id=50256
    )[0]["generated_text"]

    return out[len(prompt):].strip().split("\n")[0]


def run_t5(prompt):
    ids = t5_tok(
        prompt,
        return_tensors="pt"
    ).input_ids

    with torch.no_grad():
        out = t5.generate(
            ids,
            max_new_tokens=20
        )

    return t5_tok.decode(
        out[0],
        skip_special_tokens=True
    )


print("Model functions defined successfully!")

Model functions defined successfully!


In [7]:
tasks = [
    (
        "Fill the blank",
        "The capital of France is [MASK].",
        "The capital of France is",
        "Fill in the blank: The capital of France is ___."
    ),

    (
        "Factual QA",
        "Question: what is the largest planet? Answer: [MASK].",
        "Q: What is the largest planet?\nA:",
        "What is the largest planet?"
    ),

    (
        "Sentiment",
        "The movie was wonderful. Overall it was [MASK].",
        "Review: The movie was wonderful. Sentiment:",
        "Is this review positive or negative? The movie was wonderful."
    ),

    (
        "Translation",
        "In German, hello is [MASK].",
        "English: Good morning\nGerman:",
        "Translate English to German: Good morning"
    ),

    (
        "Summarise",
        "Transformers use attention. In short, they rely on [MASK].",
        "Transformers use self-attention to relate every token to every other token, which made RNNs obsolete. TL;DR:",
        "Summarize: Transformers use self-attention to relate every token to every other token, which made RNNs obsolete."
    )
]

print("Five tasks defined successfully!")

Five tasks defined successfully!


In [8]:
rows = []

for name, p_bert, p_gpt, p_t5 in tasks:

    row = {"task": name}

    for fam, fn, p in [
        ("BERT (encoder)", run_bert, p_bert),
        ("GPT-2 (decoder)", run_gpt2, p_gpt),
        ("Flan-T5 (enc-dec)", run_t5, p_t5)
    ]:

        start = time.time()

        row[fam] = fn(p)

        row[fam + " s"] = round(
            time.time() - start, 2
        )

    rows.append(row)

df = pd.DataFrame(rows)

print("All tasks executed successfully!")

All tasks executed successfully!


In [9]:
pd.set_option("display.max_colwidth", 45)

print("MODEL COMPARISON RESULTS\n")

print(
    df[
        [
            "task",
            "BERT (encoder)",
            "GPT-2 (decoder)",
            "Flan-T5 (enc-dec)"
        ]
    ].to_string(index=False)
)

MODEL COMPARISON RESULTS

          task BERT (encoder)                                            GPT-2 (decoder)                             Flan-T5 (enc-dec)
Fill the blank          paris the capital of the French Republic, and the capital of the                                        london
    Factual QA           mars    The largest planet is the Sun. It is the largest planet                                        uranus
     Sentiment           good                                                       5/10                                      positive
   Translation          hello                                                      Hello                                 Good morning!
     Summarise      attention              RNNs are a great way to learn about the world Transformers are a relic of the original RNN.


In [10]:
print("EXECUTION TIME (SECONDS)\n")

print(
    df[
        [
            "task",
            "BERT (encoder) s",
            "GPT-2 (decoder) s",
            "Flan-T5 (enc-dec) s"
        ]
    ].to_string(index=False)
)

EXECUTION TIME (SECONDS)

          task  BERT (encoder) s  GPT-2 (decoder) s  Flan-T5 (enc-dec) s
Fill the blank              0.39               0.79                 0.58
    Factual QA              0.13               0.89                 1.39
     Sentiment              0.45               2.34                 0.52
   Translation              0.24               1.35                 0.55
     Summarise              0.24               1.15                 1.73


In [11]:
print("OBSERVATIONS\n")

print("1. BERT is an encoder model.")
print("2. GPT-2 is a decoder model.")
print("3. Flan-T5 is an encoder-decoder model.")

print("\nKEY DIFFERENCES\n")

print("BERT: Best suited for understanding and filling masked words.")
print("GPT-2: Generates text by predicting the next token.")
print("Flan-T5: Performs instruction-based tasks such as translation and summarization.")

print("\nEXPERIMENT COMPLETED SUCCESSFULLY!")

OBSERVATIONS

1. BERT is an encoder model.
2. GPT-2 is a decoder model.
3. Flan-T5 is an encoder-decoder model.

KEY DIFFERENCES

BERT: Best suited for understanding and filling masked words.
GPT-2: Generates text by predicting the next token.
Flan-T5: Performs instruction-based tasks such as translation and summarization.

EXPERIMENT COMPLETED SUCCESSFULLY!
